In [ ]:
from pathlib import Path
%pip install pandas
import pandas as pd


def load_csv(path):
    csv_file = Path(path)

    if not csv_file.exists():
        raise FileNotFoundError(
            f"CSV file not found: {csv_file.resolve()}. Check the path and try again."
        )
    if not csv_file.is_file():
        raise ValueError(f"CSV path is not a file: {csv_file.resolve()}")

    try:
        frame = pd.read_csv(csv_file)
    except PermissionError as error:
        raise PermissionError(
            f"CSV file cannot be read due to permissions: {csv_file.resolve()}"
        ) from error
    except pd.errors.EmptyDataError as error:
        raise ValueError(f"CSV file is empty: {csv_file.resolve()}") from error
    except pd.errors.ParserError as error:
        raise ValueError(
            f"CSV file has an invalid format: {csv_file.resolve()}. Check its rows and delimiters."
        ) from error
    except OSError as error:
        raise OSError(
            f"Unable to read CSV file: {csv_file.resolve()}. {error}"
        ) from error

    if frame.empty:
        raise ValueError(f"CSV file contains no data rows: {csv_file.resolve()}")

    return frame


csv_path = Path("../RawData/vehicle_data.csv")
data = load_csv(csv_path)
data.head()

In [ ]:
import time
from datetime import datetime, timezone


def simulate_stream(frame, events_per_second=2, burst_limit=3, max_events=10):
    if events_per_second <= 0:
        raise ValueError("events_per_second must be greater than zero")
    if burst_limit < 1:
        raise ValueError("burst_limit must be at least 1")
    if max_events < 0:
        raise ValueError("max_events cannot be negative")

    rows = frame.to_dict(orient="records")
    if not rows:
        raise ValueError("The input data has no rows")

    tokens = float(burst_limit)
    last_refill = time.monotonic()

    for sequence in range(max_events):
        while tokens < 1:
            now = time.monotonic()
            tokens = min(burst_limit, tokens + (now - last_refill) * events_per_second)
            last_refill = now
            if tokens < 1:
                time.sleep((1 - tokens) / events_per_second)

        tokens -= 1
        yield {
            "device_id": "vehicle-001",
            "sequence": sequence,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "payload": rows[sequence % len(rows)],
        }


# Try a stream; the first burst_limit events can arrive immediately.
for event in simulate_stream(
    data,
    events_per_second=2,
    burst_limit=3,
    max_events=10,
):
    print(event)